In [ ]:
from dotenv import load_dotenv
from concurrent.futures import ThreadPoolExecutor, as_completed
import os
from openai import OpenAI
import pandas as pd
import importlib
import helpers
import fitment_logic
import json

importlib.reload(helpers)
importlib.reload(fitment_logic)

load_dotenv()
print("Key found:", os.getenv("OPENAI_API_KEY") is not None)
api_key = os.getenv("OPENAI_API_KEY")
PROMPT_ID = os.getenv("PROMPT_ID")
PROMPT_VERSION = os.getenv("PROMPT_VERSION")

if not api_key:
    raise Exception("OPENAI_API_KEY not found")
if not PROMPT_ID:
    raise Exception("PROMPT_ID not found")
if not PROMPT_VERSION:
    raise Exception("PROMPT_VERSION not found")

client = OpenAI(api_key=api_key)

OUTPUT_FILE = "products_first_100.xlsx"
FITMENT_FILE = "nexus_ymms.csv"
CUSTOM_FITMENT_FILE = "custom_fitments.xlsx"

fitment_df = pd.read_csv(FITMENT_FILE)
fitment_df.columns = [col.strip().lower() for col in fitment_df.columns]
fitment_df["year"] = pd.to_numeric(fitment_df["year"], errors="coerce").astype("Int64")

if os.path.exists(CUSTOM_FITMENT_FILE):
    custom_fitment_df = pd.read_excel(CUSTOM_FITMENT_FILE)
else:
    custom_fitment_df = pd.DataFrame(columns=["SKU", "Year", "Make", "Model", "Submodel", "Universal"])

TITLE_COL = "Title"
TYPE_COL = "Type"
TAGS_COL = "Tags"
COLLECTIONS_COL = "Custom Collections"
WEIGHT_COL = "Variant Weight"
REVIEW_COL = "AI Review Reasons"

FITMENT_COL = "Metafield: convermax.fitment [list.single_line_text_field]"
UNIVERSAL_COL = "Metafield: convermax.universal [boolean]"

COMMAND_COL = "Command"
TAGS_COMMAND_COL = "Tags Command"
VARIANT_COMMAND_COL = "Variant Command"

signature_columns = [TITLE_COL, TYPE_COL, TAGS_COL, COLLECTIONS_COL, FITMENT_COL, UNIVERSAL_COL, COMMAND_COL, TAGS_COMMAND_COL, VARIANT_COMMAND_COL, REVIEW_COL]

print("FINISHED setup")

Key found: True
FINISHED setup


In [52]:
MANUAL_FITMENT_OVERRIDES = {
    ("Mercedes-Benz", "A250"): [
        {"start_year": 2012, "end_year": 2024, "metadata": "Mercedes-Benz|A250"},
    ],

    ("Mercedes-Benz", "E250"): [
        {"start_year": 2009, "end_year": 2013, "metadata": "Mercedes-Benz|E250"},
        {"start_year": 2014, "end_year": 2016, "metadata": "Mercedes-Benz|E250|Bluetec"},
        {"start_year": 2014, "end_year": 2016, "metadata": "Mercedes-Benz|E250|Bluetec 4Matic"},
        {"start_year": 2017, "end_year": 2020, "metadata": "Mercedes-Benz|E250"},
    ],
    ("Mercedes-Benz", "AMG GLC43"): [
        {"start_year": 2017, "end_year": 2026, "metadata": "Mercedes-Benz|GLC43 AMG|4Matic"},
    ],
    ("Audi", "S5 Sportback"): [
        {"start_year": 2008, "end_year": 2016, "metadata": "Audi|S5 Sportback"},
    ],
    ("Mercedes-Benz", "CLS350"): [
        {"start_year": 2010, "end_year": 2016, "metadata": "Mercedes-Benz|CLS350"},
    ],
    ("Mercedes-Benz", "CLS 350"): [
        {"start_year": 2010, "end_year": 2016, "metadata": "Mercedes-Benz|CLS350"},
    ],
    ("BMW", "M140i"): [
        {"start_year": 2016, "end_year": 2019, "metadata": "BMW|M140i"}
    ],
    ("BMW", "M340i"): [
        {"start_year": 2019, "end_year": 2025, "metadata": "BMW|M340i|Base"},
    ],
    ("Aston Martin", "V8 Vantage"): [
        {"start_year": 2018, "end_year": 2023, "metadata": "Aston Martin|Vantage|Base"},
        {"start_year": 2019, "end_year": 2019, "metadata": "Aston Martin|Vantage|AMR"},
        {"start_year": 2021, "end_year": 2023, "metadata": "Aston Martin|Vantage|F1 Edition"},
        {"start_year": 2023, "end_year": 2023, "metadata": "Aston Martin|Vantage|V12"},
    ],

    ("Aston Martin", "Vantage"): [
        {"start_year": 2018, "end_year": 2023, "metadata": "Aston Martin|Vantage|Base"},
        {"start_year": 2019, "end_year": 2019, "metadata": "Aston Martin|Vantage|AMR"},
        {"start_year": 2021, "end_year": 2023, "metadata": "Aston Martin|Vantage|F1 Edition"},
        {"start_year": 2023, "end_year": 2023, "metadata": "Aston Martin|Vantage|V12"},
    ],
    ("Mercedes-Benz", "SL500"): [
        {"start_year": 2007, "end_year": 2015, "metadata": "Mercedes-Benz|SL500"},
    ],
    ("Skoda", "SuperB"): [
        {"start_year": 2015, "end_year": 2015, "metadata": "Skoda|SuperB"},
    ],
    ("BMW", "M135i"): [
        {"start_year": 2019, "end_year": 2021, "metadata": "BMW|M135i"},
    ],
    ("Audi", "S3"): [
        {"start_year": 2003, "end_year": 2014, "metadata": "Audi|S3"},
    ],
    ("BMW", "220i"): [
    {"start_year": 2014, "end_year": 2024, "metadata": "BMW|220i"},
    ],
    ("BMW", "316d"): [
        {"start_year": 2008, "end_year": 2020, "metadata": "BMW|316d"},
    ],
    ("BMW", "316i"): [
        {"start_year": 2006, "end_year": 2018, "metadata": "BMW|316i"},
    ],
    ("BMW", "318d"): [
        {"start_year": 2006, "end_year": 2020, "metadata": "BMW|318d"},
    ],
    ("BMW", "318i"): [
        {"start_year": 2006, "end_year": 2019, "metadata": "BMW|318i"},
    ],
    ("BMW", "320d"): [
        {"start_year": 2006, "end_year": 2021, "metadata": "BMW|320d"},
    ],
    ("BMW", "325d"): [
        {"start_year": 2005, "end_year": 2021, "metadata": "BMW|325d"},
    ],
    ("BMW", "325i"): [
        {"start_year": 2007, "end_year": 2018, "metadata": "BMW|325i"},
    ],
    ("BMW", "330d"): [
        {"start_year": 2006, "end_year": 2021, "metadata": "BMW|330d"},
    ],
    ("BMW", "335d"): [
        {"start_year": 2006, "end_year": 2019, "metadata": "BMW|335d"},
    ],
    ("BMW", "335is"): [
        {"start_year": 2012, "end_year": 2018, "metadata": "BMW|335is"},
    ],
    ("BMW", "420d"): [
        {"start_year": 2015, "end_year": 2020, "metadata": "BMW|420d"},
    ],
    ("BMW", "420i"): [
        {"start_year": 2014, "end_year": 2023, "metadata": "BMW|420i"},
    ],
    ("Ferrari", "GTC4Lusso"): [
        {"start_year": 2016, "end_year": 2016, "metadata": "Ferrari|GTC4Lusso"},
    ],
    ("BMW", "430d"): [{"start_year": 2019, "end_year": 2020, "metadata": "BMW|430d"}],
("BMW", "435d"): [{"start_year": 2019, "end_year": 2020, "metadata": "BMW|435d"}],
("BMW", "520d"): [{"start_year": 2011, "end_year": 2020, "metadata": "BMW|520d"}],
("BMW", "520i"): [{"start_year": 2011, "end_year": 2022, "metadata": "BMW|520i"}],
("BMW", "525d"): [{"start_year": 2016, "end_year": 2020, "metadata": "BMW|525d"}],
("BMW", "528i"): [{"start_year": 2017, "end_year": 2018, "metadata": "BMW|528i"}],
("BMW", "530d"): [{"start_year": 2016, "end_year": 2020, "metadata": "BMW|530d"}],
("BMW", "535i"): [{"start_year": 2017, "end_year": 2019, "metadata": "BMW|535i"}],
("BMW", "540d"): [{"start_year": 2017, "end_year": 2020, "metadata": "BMW|540d"}],
("BMW", "550i"): [{"start_year": 2017, "end_year": 2022, "metadata": "BMW|550i"}],
("BMW", "640i"): [{"start_year": 2019, "end_year": 2023, "metadata": "BMW|640i"}],
("BMW", "650i"): [{"start_year": 2011, "end_year": 2011, "metadata": "BMW|650i"}],
("BMW", "725d"): [{"start_year": 2019, "end_year": 2019, "metadata": "BMW|725d"}],
("BMW", "725Ld"): [{"start_year": 2019, "end_year": 2019, "metadata": "BMW|725Ld"}],
("BMW", "730d"): [{"start_year": 2019, "end_year": 2019, "metadata": "BMW|730d"}],
("BMW", "730i"): [{"start_year": 2016, "end_year": 2019, "metadata": "BMW|730i"}],
("BMW", "730Ld"): [{"start_year": 2019, "end_year": 2019, "metadata": "BMW|730Ld"}],
("BMW", "740d"): [{"start_year": 2019, "end_year": 2019, "metadata": "BMW|740d"}],
("BMW", "740e"): [{"start_year": 2016, "end_year": 2019, "metadata": "BMW|740e"}],
("BMW", "740Ld"): [{"start_year": 2019, "end_year": 2019, "metadata": "BMW|740Ld"}],
("BMW", "740Li"): [{"start_year": 2016, "end_year": 2021, "metadata": "BMW|740Li"}],
("BMW", "750d"): [{"start_year": 2017, "end_year": 2019, "metadata": "BMW|750d"}],
("BMW", "750Ld"): [{"start_year": 2019, "end_year": 2019, "metadata": "BMW|750Ld"}],
("BMW", "750Li"): [{"start_year": 2019, "end_year": 2019, "metadata": "BMW|750Li"}],
("BMW", "760i"): [{"start_year": 2019, "end_year": 2019, "metadata": "BMW|760i"}],
("BMW", "760Li"): [{"start_year": 2019, "end_year": 2019, "metadata": "BMW|760Li"}],
("BMW", "M550d"): [{"start_year": 2017, "end_year": 2020, "metadata": "BMW|M550d"}],
("BMW", "M760i"): [{"start_year": 2019, "end_year": 2019, "metadata": "BMW|M760i"}],
("BMW", "M760Li"): [{"start_year": 2017, "end_year": 2019, "metadata": "BMW|M760Li"}],
("Audi", "RS4"): [{"start_year": 2017, "end_year": 2023, "metadata": "Audi|RS4"},],
("Audi", "RS5"): [{"start_year": 2020, "end_year": 2020, "metadata": "Audi|RS5"},],
("Audi", "R8"): [{"start_year": 2013, "end_year": 2013, "metadata": "Audi|R8"},],

}

In [66]:
df = pd.read_excel(OUTPUT_FILE)

START_ROW = 0
NUM_PRODUCTS_TO_PROCESS = 10
MAX_WORKERS = 10

for col in signature_columns:
    if col not in df.columns: df[col] = None
    df[col] = df[col].astype("object")

df[WEIGHT_COL] = pd.to_numeric(df[WEIGHT_COL], errors="coerce")

rows_to_process = list(df.iloc[START_ROW:START_ROW + NUM_PRODUCTS_TO_PROCESS].iterrows())

api_results = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [
        executor.submit(helpers.safe_call_ai_for_row, client, PROMPT_ID, PROMPT_VERSION, row_index, row)
        for row_index, row in rows_to_process
    ]

    for future in as_completed(futures):
        api_results.append(future.result())

for item in sorted(api_results, key=lambda x: x["row_index"]):
    row_index = item["row_index"]
    ai = item["ai_result"]
    original_row = df.loc[row_index]

    if ai.get("convermax_universal") is True:
        final_metadata = ["Universal"]
        fitment_tag = "fits_Universal"
        fitment_review_notes = []
    else:
        final_metadata, custom_fitment_df, fitment_review_notes = fitment_logic.resolve_vehicle_fitments_to_metadata(
            fitment_df=fitment_df,
            vehicle_fitments=ai.get("vehicle_fitments", []),
            custom_fitment_df=custom_fitment_df,
            custom_fitment_file=CUSTOM_FITMENT_FILE,
            product_row=original_row,
        )
        fitment_tag = fitment_logic.metadata_to_fitment_tag(final_metadata)

    review_notes = []

    if ai.get("review_required") is True:
        review_notes.extend(ai.get("review_reasons", []))

    review_notes.extend(fitment_review_notes)

    # TODO: This should never happen though!!!!!
    if ai.get("convermax_universal") is not True and ai.get("vehicle_fitments") and not final_metadata:
        review_notes.append("fitment_metadata_empty_for_vehicle_specific_product")

    review_notes, ignored_review_notes = fitment_logic.filter_review_notes(review_notes)

    if ignored_review_notes:
        print(
            f"Ignored review reason(s) for SKU {original_row['Variant SKU']}: "
            f"{', '.join(ignored_review_notes)}"
        )

    df.at[row_index, REVIEW_COL] = " | ".join(review_notes) if review_notes else None

    existing_tags = helpers.parse_tags(original_row[TAGS_COL])

    if fitment_tag and fitment_tag not in existing_tags:
        existing_tags.append(fitment_tag)

    df.at[row_index, COMMAND_COL] = "MERGE"
    df.at[row_index, TAGS_COMMAND_COL] = "REPLACE"
    df.at[row_index, VARIANT_COMMAND_COL] = "MERGE"
    df.at[row_index, TITLE_COL] = ai.get("title", original_row[TITLE_COL])
    df.at[row_index, TYPE_COL] = ai.get("type", original_row[TYPE_COL])
    df.at[row_index, TAGS_COL] = helpers.format_tags(existing_tags)
    df.at[row_index, COLLECTIONS_COL] = ", ".join(ai.get("custom_collections", []))
    df.at[row_index, WEIGHT_COL] = ai.get("variant_weight", original_row[WEIGHT_COL])
    df.at[row_index, FITMENT_COL] = json.dumps(final_metadata, ensure_ascii=False)
    df.at[row_index, UNIVERSAL_COL] = "Yes" if ai.get("convermax_universal") is True else None

    print("\n" + "=" * 100)
    print(f"ROW: {row_index}")
    print(f"ID: {original_row['ID']}")
    print(f"OLD TITLE: {original_row[TITLE_COL]}")
    print(f"NEW TITLE: {df.at[row_index, TITLE_COL]}")
    print(f"TYPE: {df.at[row_index, TYPE_COL]}")
    print(f"COLLECTIONS: {df.at[row_index, COLLECTIONS_COL]}")
    print(f"WEIGHT: {df.at[row_index, WEIGHT_COL]}")
    print(f"METADATA: {final_metadata}")
    print(f"FITMENT TAG: {fitment_tag}")

    if ai.get("review_required") is True:
        print(f"AI REVIEW REASONS: {ai.get('review_reasons', [])}")

df.to_excel(OUTPUT_FILE, index=False)
helpers.format_products_workbook(OUTPUT_FILE)

flagged_count = df[REVIEW_COL].notna().sum()
print(f"\nProducts requiring review: {flagged_count}")

print(f"\nDone. Saved updated file to: {OUTPUT_FILE}")

Saved 1 custom fitment(s) for SKU FR-458-HDS4
Ignored review reason(s) for SKU FR-458-HDS4: title_price_description_scope_conflict

ROW: 0
ID: 8455264502046
OLD TITLE: FI Exhaust Equal Length Header Ferrari 458 Italia / Spyder 2010+
NEW TITLE: FI Exhaust Equal Length Header Ferrari 458 Italia / 458 Spider 2010-2015
TYPE: Headers & Manifolds
COLLECTIONS: Headers & Manifolds, Ferrari
WEIGHT: 28
METADATA: ['2010-2015|Ferrari|458 Italia|Base', '2010-2015|Ferrari|458 Spider|Base', '2012-2015|Ferrari|458 Spider|Base', '2015|Ferrari|458 Speciale|Base']
FITMENT TAG: fits_2010-2015`Ferrari`458 Italia`Base~2010-2015`Ferrari`458 Spider`Base~2012-2015`Ferrari`458 Spider`Base~2015`Ferrari`458 Speciale`Base
AI REVIEW REASONS: ['title_price_description_scope_conflict']
Ignored review reason(s) for SKU TIP-SPR-TBS: title_price_description_scope_conflict

ROW: 1
ID: 8455264567582
OLD TITLE: FI Exhaust Dual Tips Titanium Toyota Supra 2020+
NEW TITLE: FI Exhaust Titanium Dual Tips Toyota Supra 2020-2023


In [61]:
import pandas as pd

INPUT_FILE = "SignatureAutohausExport_2026-06-09_181952.xlsx"
OUTPUT_100_FILE = "products_first_100.xlsx"

df_100 = pd.read_excel(INPUT_FILE).head(100)

df_100.to_excel(OUTPUT_100_FILE, index=False)

print(f"Saved first 100 rows to {OUTPUT_100_FILE}")

Saved first 100 rows to products_first_100.xlsx


In [67]:
import pandas as pd

INPUT_FILE = "products_first_100.xlsx"
OUTPUT_FILE_10 = "products_first_10_upload_test.xlsx"

df = pd.read_excel(INPUT_FILE)

# Keep first 10 product rows.
# Excel header is not counted as a DataFrame row, so this creates:
# 1 header row + 10 product rows = 11 Excel rows total.
df_upload = df.head(10).copy()

# Remove AI Review Reasons column before upload.
if "AI Review Reasons" in df_upload.columns:
    df_upload = df_upload.drop(columns=["AI Review Reasons"])

df_upload.to_excel(OUTPUT_FILE_10, index=False)

print(f"Saved upload test file: {OUTPUT_FILE_10}")
print(f"Rows in file excluding header: {len(df_upload)}")

Saved upload test file: products_first_10_upload_test.xlsx
Rows in file excluding header: 10


In [ ]:
TEST_FILE = "SignatureAutohausExport_2026-06-09_181952.xlsx"
TEST_ROW_INDEX = 2

signature_df = pd.read_excel(TEST_FILE)

row = signature_df.loc[TEST_ROW_INDEX]
product = build_product_input(row)

print("INPUT PRODUCT:")
print(json.dumps(product, indent=2, ensure_ascii=False))

response = client.responses.create(
    prompt={
        "id": PROMPT_ID,
        "version": PROMPT_VERSION,
        "variables": {
            "product": json.dumps(product, ensure_ascii=False)
        }
    }
)

ai_result = json.loads(response.output_text)

print("\nAI RESPONSE:")
print(json.dumps(ai_result, indent=2, ensure_ascii=False))

if ai_result.get("convermax_universal") is True:
    final_metadata = []
    fitment_tag = ""
    lookup_notes = []
else:
    final_metadata, lookup_notes = resolve_vehicle_fitments_to_metadata(
        ai_result.get("vehicle_fitments", [])
    )
    fitment_tag = metadata_to_fitment_tag(final_metadata)

print("\nPYTHON FITMENT RESULT:")
print("METADATA:")
print(json.dumps(final_metadata, indent=2, ensure_ascii=False))

print("\nFITMENT TAG:")
print(fitment_tag)

print("\nLOOKUP NOTES:")
print(json.dumps(lookup_notes, indent=2, ensure_ascii=False))

INPUT PRODUCT:
{
  "title": "FI Exhaust Catback Valvetronic Muffler with Quad Tips BMW X6M 15-18",
  "body_html": "<div id=\"productDescription\">\n                            <p id=\"products_description\">Frequency Intelligent Valvetronic Exhaust System technology offers cutting-edge intelligent ECU exhaust control valve, with an emphasis on the optimization of both acoustics and performance. It is a testament to their philosophy of the ultimate union of comfort and performance experience for the driver and passengers.<br><br>When the valves are fully open for maximum flow and power, it creates an exotic tone and allows for high performance. When the valves are closed, volume is reduced for a more low-profile comfortable drive. With their latest technology, just one click on the remote control will setup any rpm to automatic mode. The automatic mode enables the system to detect the engine RPM to intelligently switch comfort/racing exhaust profiles. Users can also opt to simply switch

In [10]:
BMW_MODELS = [
    "316d",
    "318d",
    "318i",
    "320d",
    "320i",
    "325d",
    "330d",
    "330e",
    "330i",
    "335d",
    "335i",
    "340i",
    "420d",
    "420i",
    "430d",
    "430i",
    "435d",
    "440i",
    "520d",
    "520i",
    "525d",
    "528i",
    "530d",
    "530e",
    "530i",
    "535i",
    "540d",
    "540i",
    "550i",
    "630d",
    "630i",
    "640i",
    "650i",
    "725d",
    "725Ld",
    "730d",
    "730i",
    "730Ld",
    "740d",
    "740e",
    "740i",
    "740Ld",
    "740Li",
    "745e",
    "750d",
    "750i",
    "750Ld",
    "750Li",
    "760i",
    "760Li",
    "840d",
    "840i",
    "M340i",
    "M550d",
    "M550i",
    "M6",
    "M760i",
    "M760Li",
    "M8",
    "M850i",
    "X3",
    "X3 M",
    "X4",
    "X4 M",
    "X5",
    "X5 M",
    "X6",
    "X6 M",
    "X7",
    "Z4",
]

YEARS = range(2018, 2026)

found = []

missing = []

for model in BMW_MODELS:

    for year in YEARS:

        matches = fitment_df[

            (fitment_df["make"] == "BMW")

            & (fitment_df["model"] == model)

            & (fitment_df["year"] == year)

        ]

        if matches.empty:

            missing.append(f"{year}|BMW|{model}")

            continue

        for _, row in matches.iterrows():

            submodel = str(row["submodel"]).strip()

            metadata = (

                f"{year}|BMW|{model}"

                if not submodel

                else f"{year}|BMW|{model}|{submodel}"

            )

            found.append(metadata)

found = sorted(set(found))

missing = sorted(set(missing))

print("\nFOUND METADATA:\n")

for m in found:

    print(m)

print("\nMISSING FITMENTS:\n")

for m in missing:

    print(m)


FOUND METADATA:

2018|BMW|320i|Base
2018|BMW|330e|Base
2018|BMW|330i|Base
2018|BMW|340i|Base
2018|BMW|430i|Base
2018|BMW|440i|Base
2018|BMW|530e|Base
2018|BMW|530i|Base
2018|BMW|540i|Base
2018|BMW|640i|Base
2018|BMW|650i|Base
2018|BMW|740i|Base
2018|BMW|750i|Base
2018|BMW|M6|Base
2018|BMW|X3|M40i
2018|BMW|X3|xDrive30i
2018|BMW|X4|M40i
2018|BMW|X4|xDrive28i
2018|BMW|X5|M
2018|BMW|X5|sDrive35i
2018|BMW|X5|xDrive35d
2018|BMW|X5|xDrive35i
2018|BMW|X5|xDrive40e
2018|BMW|X5|xDrive50i
2018|BMW|X6|M
2018|BMW|X6|sDrive35i
2018|BMW|X6|xDrive35i
2018|BMW|X6|xDrive50i
2019|BMW|330i|Base
2019|BMW|430i|Base
2019|BMW|440i|Base
2019|BMW|530e|Base
2019|BMW|530i|Base
2019|BMW|540i|Base
2019|BMW|740i|Base
2019|BMW|750i|Base
2019|BMW|X3|M40i
2019|BMW|X3|sDrive30i
2019|BMW|X3|xDrive30i
2019|BMW|X4|M40i
2019|BMW|X4|xDrive30i
2019|BMW|X5|xDrive40i
2019|BMW|X5|xDrive50i
2019|BMW|X6|M
2019|BMW|X6|sDrive35i
2019|BMW|X6|xDrive35i
2019|BMW|X6|xDrive50i
2019|BMW|X7|xDrive40i
2019|BMW|X7|xDrive50i
2019|BMW|Z4|sDri

In [14]:
def metadata_to_fitment_tag(metadata_list):
    return "fits_" + "~".join(
        m.replace("|", "`") for m in metadata_list
    )

metadata = ["2015-2020|BMW|316d","2015-2020|BMW|318d","2015-2019|BMW|318i","2015-2021|BMW|320d","2018|BMW|320i|Base","2015-2021|BMW|325d","2015-2021|BMW|330d","2018|BMW|330e|Base","2021-2022|BMW|330e|Base","2018-2022|BMW|330i|Base","2015-2019|BMW|335d","2018|BMW|335i|Base","2018|BMW|340i|Base","2015-2020|BMW|420d","2015-2022|BMW|420i","2019-2020|BMW|430d","2018-2022|BMW|430i|Base","2019-2020|BMW|435d","2018-2020|BMW|440i|Base","2015-2020|BMW|520d","2015-2022|BMW|520i","2016-2020|BMW|525d","2017-2018|BMW|528i","2016-2020|BMW|530d","2018-2022|BMW|530e|Base","2018-2022|BMW|530i|Base","2017-2019|BMW|535i","2017-2020|BMW|540d","2018-2022|BMW|540i|Base","2017-2022|BMW|550i","2019-2022|BMW|640i","2018|BMW|640i|Base","2018|BMW|650i|Base","2019|BMW|725d","2019|BMW|725Ld","2019|BMW|730d","2016-2019|BMW|730i","2019|BMW|730Ld","2019|BMW|740d","2016-2019|BMW|740e","2018-2022|BMW|740i|Base","2019|BMW|740Ld","2016-2021|BMW|740Li","2017-2019|BMW|750d","2018-2019|BMW|750i|Base","2019|BMW|750Ld","2019|BMW|750Li","2019|BMW|760i","2019|BMW|760Li","2020-2022|BMW|840i|Base","2019-2022|BMW|M340i|Base","2017-2020|BMW|M550d","2018-2022|BMW|M550i xDrive|Base","2018|BMW|M6|Base","2019|BMW|M760i","2017-2019|BMW|M760Li","2020|BMW|M8|Base","2020|BMW|M8|Competition","2022|BMW|M8|Competition","2019-2022|BMW|M850i xDrive|Base","2020-2022|BMW|M850i xDrive Gran Coupe|Base","2020-2022|BMW|X3|M","2020-2022|BMW|X3|M Competition","2018-2022|BMW|X3|M40i","2019-2022|BMW|X3|sDrive30i","2020-2021|BMW|X3|xDrive30e","2018-2022|BMW|X3|xDrive30i","2020-2022|BMW|X4|M","2020-2022|BMW|X4|M Competition","2018-2022|BMW|X4|M40i","2018|BMW|X4|xDrive28i","2019-2022|BMW|X4|xDrive30i","2018-2022|BMW|X5|M","2020|BMW|X5|M Competition","2020-2022|BMW|X5|M50i","2018|BMW|X5|sDrive35i","2020-2022|BMW|X5|sDrive40i","2018|BMW|X5|xDrive35d","2018|BMW|X5|xDrive35i","2018|BMW|X5|xDrive40e","2019-2022|BMW|X5|xDrive40i","2021-2022|BMW|X5|xDrive45e","2018-2020|BMW|X5|xDrive50i","2018-2022|BMW|X6|M","2020-2021|BMW|X6|M Competition","2020-2022|BMW|X6|M50i","2018-2019|BMW|X6|sDrive35i","2020-2021|BMW|X6|sDrive40i","2018-2019|BMW|X6|xDrive35i","2020-2022|BMW|X6|xDrive40i","2018-2019|BMW|X6|xDrive50i","2020-2022|BMW|X7|M50i","2019-2022|BMW|X7|xDrive40i","2019-2020|BMW|X7|xDrive50i","2022|BMW|Z4|M40i","2020|BMW|Z4|M40i First Edition","2020-2021|BMW|Z4|sDrive M40i","2019-2022|BMW|Z4|sDrive30i"]

fitment_tag = metadata_to_fitment_tag(metadata)

print(fitment_tag)


fits_2015-2020`BMW`316d~2015-2020`BMW`318d~2015-2019`BMW`318i~2015-2021`BMW`320d~2018`BMW`320i`Base~2015-2021`BMW`325d~2015-2021`BMW`330d~2018`BMW`330e`Base~2021-2022`BMW`330e`Base~2018-2022`BMW`330i`Base~2015-2019`BMW`335d~2018`BMW`335i`Base~2018`BMW`340i`Base~2015-2020`BMW`420d~2015-2022`BMW`420i~2019-2020`BMW`430d~2018-2022`BMW`430i`Base~2019-2020`BMW`435d~2018-2020`BMW`440i`Base~2015-2020`BMW`520d~2015-2022`BMW`520i~2016-2020`BMW`525d~2017-2018`BMW`528i~2016-2020`BMW`530d~2018-2022`BMW`530e`Base~2018-2022`BMW`530i`Base~2017-2019`BMW`535i~2017-2020`BMW`540d~2018-2022`BMW`540i`Base~2017-2022`BMW`550i~2019-2022`BMW`640i~2018`BMW`640i`Base~2018`BMW`650i`Base~2019`BMW`725d~2019`BMW`725Ld~2019`BMW`730d~2016-2019`BMW`730i~2019`BMW`730Ld~2019`BMW`740d~2016-2019`BMW`740e~2018-2022`BMW`740i`Base~2019`BMW`740Ld~2016-2021`BMW`740Li~2017-2019`BMW`750d~2018-2019`BMW`750i`Base~2019`BMW`750Ld~2019`BMW`750Li~2019`BMW`760i~2019`BMW`760Li~2020-2022`BMW`840i`Base~2019-2022`BMW`M340i`Base~2017-2020`BMW

In [39]:
test_cases = [

    ("BMW", "M760Li", 2017, 2021),

    ("Audi", "RS4", 2017, 2025),

    ("Audi", "RS5", 2020, 2025),

    ("Audi", "R8", 2013, 2017),

]

for make, model, start_year, end_year in test_cases:

    matches = get_matching_csv_rows(

        fitment_df=fitment_df,

        start_year=start_year,

        end_year=end_year,

        make=make,

        model=model,

        submodel=""

    )

    print("=" * 80)

    print(f"{start_year}-{end_year} {make} {model}")

    print(f"Matches found: {len(matches)}")

    display(matches)

2017-2021 BMW M760Li
Matches found: 0


,year,make,model,submodel,url


2017-2025 Audi RS4
Matches found: 0


,year,make,model,submodel,url


2020-2025 Audi RS5
Matches found: 4


,year,make,model,submodel,url
3033,2021,Audi,RS5,Base,"AFE35-10027C,AFE77-92005,AFE77-92005-MC,AWE266..."
3034,2022,Audi,RS5,Base,"AFE35-10027C,AFE77-92005,AFE77-92005-MC,AWE266..."
3035,2023,Audi,RS5,Base,"AFE35-10027C,AFE77-92005,AFE77-92005-MC,AWE266..."
3036,2024,Audi,RS5,Base,"AFE35-10027C,AWE2660-15032,AWE3015-33123,AWE30..."


2013-2017 Audi R8
Matches found: 9


,year,make,model,submodel,url
2972,2014,Audi,R8,Base,"AFE10-10408DM,AFE10-10408RM,AFE57-10012R,AFE77..."
2973,2015,Audi,R8,Base,"AFE10-10408DM,AFE10-10408RM,AFE57-10012R,AFE77..."
2974,2017,Audi,R8,Base,"AFE57-10012R,AFE77-20008,AKRP-HF946,AKRTP-CT/3..."
2992,2014,Audi,R8,Plus,"AFE57-10012R,AFE77-16410,AFE77-20008,AKRP-HF88..."
2993,2015,Audi,R8,Plus,"AFE57-10012R,AFE77-16410,AFE77-20008,AKRP-HF88..."
2994,2017,Audi,R8,Plus,"AFE57-10012R,AFE77-20008,AKRP-HF946,AKRTP-CT/3..."
3000,2014,Audi,R8,Spyder,"AFE10-10408DM,AFE10-10408RM,AFE57-10012R,AFE77..."
3001,2015,Audi,R8,Spyder,"AFE10-10408DM,AFE10-10408RM,AFE57-10012R,AFE77..."
3002,2017,Audi,R8,Spyder,"AFE57-10012R,AFE77-20008,AKRP-HF946,AKRTP-CT/3..."
